In [1]:
!pip install timm datasets transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 48.6 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [2]:
import pandas as pd

# Standard Kaggle path configuration for CheXpert small
metadata_path = '/kaggle/input/datasets/ashery/chexpert/valid.csv'

df = pd.read_csv(metadata_path)
print(df.columns)

Index(['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding',
       'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
       'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
       'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
       'Support Devices'],
      dtype='object')


In [3]:
# Define label columns
label_columns = [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
    ]

# Extract image paths and labels
df['Path'] = df['Path'].apply(lambda x: f"/kaggle/input/chexpert/{x}")  # Ensure correct path
labels = df[label_columns].fillna(0).astype(int)  # Replace NaNs and convert to integers


In [4]:
df[label_columns] = df[label_columns].replace(-1, 0)


In [5]:
df[label_columns] = df[label_columns].replace(-1, 1)


In [6]:
df[label_columns]

,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
230,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
231,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
232,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Combine image paths and labels
dataset = pd.concat([df['Path'], labels], axis=1)



In [8]:
dataset.head()


,Path,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,/kaggle/input/chexpert/CheXpert-v1.0-small/val...,0,1,1,1,0,0,0,0,0,0,0,0,0,0
1,/kaggle/input/chexpert/CheXpert-v1.0-small/val...,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,/kaggle/input/chexpert/CheXpert-v1.0-small/val...,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,/kaggle/input/chexpert/CheXpert-v1.0-small/val...,0,1,0,1,0,1,0,0,0,0,0,0,0,0
4,/kaggle/input/chexpert/CheXpert-v1.0-small/val...,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [9]:
import pandas as pd

# Load metadata
metadata_path = '/kaggle/input/datasets/ashery/chexpert/train.csv'
df = pd.read_csv(metadata_path)

# Ensure all label columns are numeric and replace invalid values
# label_columns = ['Cardiomegaly', 'Edema', 'Consolidation', 'Pneumonia', 'No Finding']
df[label_columns] = df[label_columns].apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)

df['Path'] = df['Path'].str.replace(
    'CheXpert-v1.0-small/', '', regex=False
)  # Remove the incorrect prefix

# Ensure paths are correct for Kaggle directory structure
df['Path'] = df['Path'].apply(lambda x: f"/kaggle/input/chexpert/{x}")

# Combine paths and labels
dataset = pd.concat([df['Path'], df[label_columns]], axis=1)



In [10]:
import torch

# Assuming 'dataset' is your pandas dataframe containing the training data
# Columns: Path, No Finding, Enlarged Cardiomediastinum, Cardiomegaly, ... (14 classes)

# Get the label columns (skip the 'Path' column)
labels_df = dataset.iloc[:, 1:]

# Optional but recommended: Apply U-Zeros to the dataframe calculation so weights reflect what the model sees
labels_df = labels_df.replace(-1.0, 0.0)

# Calculate totals
positive_counts = labels_df.sum(axis=0).values
total_samples = len(labels_df)
negative_counts = total_samples - positive_counts

# Calculate pos_weight (add a tiny epsilon to prevent division by zero if a class is entirely empty)
epsilon = 1e-7
pos_weights_array = negative_counts / (positive_counts + epsilon)

# Convert to a PyTorch tensor and move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weights = torch.tensor(pos_weights_array, dtype=torch.float32).to(device)

print("Calculated pos_weights for 14 classes:")
print(pos_weights)

Calculated pos_weights for 14 classes:
tensor([ 8.9823, 19.6903,  7.2746,  1.1160, 23.3211,  3.2762, 14.1129, 35.9952,
         5.6939, 10.4878,  1.5922, 62.4158, 23.7139,  0.9260], device='cuda:0')


In [11]:

# Load metadata
metadata_path = '/kaggle/input/datasets/ashery/chexpert/valid.csv'
df = pd.read_csv(metadata_path)

# Ensure all label columns are numeric and replace invalid values
df[label_columns] = df[label_columns].apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)

df['Path'] = df['Path'].str.replace(
    'CheXpert-v1.0-small/', '', regex=False
)  # Remove the incorrect prefix

# Ensure paths are correct for Kaggle directory structure
df['Path'] = df['Path'].apply(lambda x: f"/kaggle/input/chexpert/{x}")

# Combine paths and labels
dataset_valid = pd.concat([df['Path'], df[label_columns]], axis=1)

In [12]:
print(dataset.head())  # Preview the DataFrame

                                                Path  No Finding  \
0  /kaggle/input/chexpert/train/patient00001/stud...         1.0   
1  /kaggle/input/chexpert/train/patient00002/stud...         0.0   
2  /kaggle/input/chexpert/train/patient00002/stud...         0.0   
3  /kaggle/input/chexpert/train/patient00002/stud...         0.0   
4  /kaggle/input/chexpert/train/patient00003/stud...         0.0   

   Enlarged Cardiomediastinum  Cardiomegaly  Lung Opacity  Lung Lesion  Edema  \
0                         0.0           0.0           0.0          0.0    0.0   
1                         0.0          -1.0           1.0          0.0   -1.0   
2                         0.0           0.0           1.0          0.0    0.0   
3                         0.0           0.0           1.0          0.0    0.0   
4                         0.0           0.0           0.0          0.0    1.0   

   Consolidation  Pneumonia  Atelectasis  Pneumothorax  Pleural Effusion  \
0            0.0        0.0 

In [13]:
print(dataset_valid.head())

                                                Path  No Finding  \
0  /kaggle/input/chexpert/valid/patient64541/stud...         0.0   
1  /kaggle/input/chexpert/valid/patient64542/stud...         0.0   
2  /kaggle/input/chexpert/valid/patient64542/stud...         0.0   
3  /kaggle/input/chexpert/valid/patient64543/stud...         0.0   
4  /kaggle/input/chexpert/valid/patient64544/stud...         1.0   

   Enlarged Cardiomediastinum  Cardiomegaly  Lung Opacity  Lung Lesion  Edema  \
0                         1.0           1.0           1.0          0.0    0.0   
1                         0.0           0.0           0.0          0.0    0.0   
2                         0.0           0.0           0.0          0.0    0.0   
3                         1.0           0.0           1.0          0.0    1.0   
4                         0.0           0.0           0.0          0.0    0.0   

   Consolidation  Pneumonia  Atelectasis  Pneumothorax  Pleural Effusion  \
0            0.0        0.0 

In [14]:
#U zeros is implemted here

import os
import torch
from torch.utils.data import Dataset
from PIL import Image

class CheXpertDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        raw_path = row['Path']
        
        # Clean the string to extract just the relative path
        if raw_path.startswith('CheXpert-v1.0-small/'):
            relative_path = raw_path.replace('CheXpert-v1.0-small/', '')
        elif raw_path.startswith('/kaggle/input/chexpert/'):
            relative_path = raw_path.replace('/kaggle/input/chexpert/', '')
        else:
            relative_path = raw_path
            
        # Strip leading slashes so os.path.join operates correctly
        relative_path = relative_path.lstrip('/')
        
        # Append to your verified working directory
        image_path = os.path.join('/kaggle/input/datasets/ashery/chexpert/', relative_path)
        
        # Load the image
        image = Image.open(image_path).convert('RGB')
        
        # Convert labels to a float tensor
        label = torch.tensor(row[1:].values.astype(float), dtype=torch.float32)
        
        # U-Zeros Strategy: Replace all -1 (uncertain) with 0 (negative)
        label[label == -1.0] = 0.0
        
        # Apply transformations
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [15]:
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize for ViT
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize
])

transform_valid = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize for ViT
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize
])



In [16]:
train_dataset = CheXpertDataset(dataframe=dataset, transform=transform)
valid_dataset = CheXpertDataset(dataframe=dataset_valid, transform=transform)

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2)

# Example: Iterate over the DataLoader
for images, labels in train_loader:
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    break

for images, labels in valid_loader:
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    break

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32, 14])
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32, 14])


In [17]:
from transformers import ViTForImageClassification, ViTImageProcessor
from torch.optim import AdamW
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch

# Load pretrained ViT
# model_name = "google/vit-base-patch16-224"
checkpoint_path = "/kaggle/input/datasets/thageo/vit-weight-v1/vit-chest-xray-epoch-2"

model = ViTForImageClassification.from_pretrained(
    checkpoint_path, 
    num_labels=14, 
    ignore_mismatched_sizes=True
)
# Processor for ViT
processor = ViTImageProcessor.from_pretrained(checkpoint_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=3e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

# Loss function
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: /kaggle/input/datasets/thageo/vit-weight-v1/vit-chest-xray-epoch-2
Key               | Status   |                                                                                      
------------------+----------+--------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([5]) vs model:torch.Size([14])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([5, 768]) vs model:torch.Size([14, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [18]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score

def train_model(model, train_loader, valid_loader, optimizer, scheduler, criterion, device, epochs=5):
    scaler = torch.amp.GradScaler()
    model.to(device)

    # Class names for CheXpert Competition subset
    class_names = [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
    ]

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

        # =======================
        # TRAINING PHASE
        # =======================
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images)
                logits = outputs.logits if hasattr(outputs, "logits") else outputs
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        print(f"Training Loss: {train_loss:.4f}")

        # =======================
        # VALIDATION PHASE
        # =======================
        model.eval()
        valid_loss = 0.0
        
        # Lists to store all labels and predictions for the epoch
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)

                with torch.amp.autocast(device_type="cuda"):
                    outputs = model(images)
                    logits = outputs.logits if hasattr(outputs, "logits") else outputs
                    loss = criterion(logits, labels)

                valid_loss += loss.item()

                # Apply sigmoid to convert logits to probabilities [0, 1]
                probs = torch.sigmoid(logits)
                
                # Move to CPU and numpy for scikit-learn
                all_labels.append(labels.cpu().numpy())
                all_preds.append(probs.cpu().numpy())

        valid_loss /= len(valid_loader)
        
        # Concatenate all batches
        all_labels = np.vstack(all_labels)
        all_preds = np.vstack(all_preds)

        # Calculate AUROC
        try:
            # Macro AUROC gives equal weight to all classes regardless of imbalance
            macro_auroc = roc_auc_score(all_labels, all_preds, average='macro')
            # Per-class AUROC
            per_class_auroc = roc_auc_score(all_labels, all_preds, average=None)
            
            print(f"Validation Loss: {valid_loss:.4f} | Macro AUROC: {macro_auroc:.4f}")
            for i, name in enumerate(class_names):
                print(f"  - {name}: {per_class_auroc[i]:.4f}")
        except ValueError as e:
            # This triggers if a class has only 1s or only 0s in the validation set
            print(f"Validation Loss: {valid_loss:.4f} | AUROC Error: {e}")

        # =======================
        # SCHEDULER & CHECKPOINTING
        # =======================
        if scheduler is not None:
            scheduler.step()

        save_path = f"/kaggle/working/vit-chest-xray-full-labels-epoch-{epoch+1}"
        print(f"Saving checkpoint to {save_path}...")
        
        # Save model and processor explicitly
        model.save_pretrained(save_path)
        
        # If your processor object is accessible globally, save it. Otherwise, initialize and save it here.
        try:
            processor.save_pretrained(save_path)
        except NameError:
            pass # Ensure processor is defined in the outer scope


In [19]:
train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    scheduler=None,  # Pass `None` if not using a scheduler
    criterion=criterion,
    device=device,
    epochs=5
)


--- Epoch 1/5 ---
Training Loss: 0.9415
Validation Loss: 1.4472 | Macro AUROC: nan
  - No Finding: 0.8804
  - Enlarged Cardiomediastinum: 0.5764
  - Cardiomegaly: 0.8324
  - Lung Opacity: 0.9039
  - Lung Lesion: 0.1803
  - Edema: 0.9058
  - Consolidation: 0.9154
  - Pneumonia: 0.5890
  - Atelectasis: 0.7790
  - Pneumothorax: 0.7945
  - Pleural Effusion: 0.9188
  - Pleural Other: 0.9614
  - Fracture: nan
  - Support Devices: 0.8128
Saving checkpoint to /kaggle/working/vit-chest-xray-full-labels-epoch-1...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Epoch 2/5 ---
Training Loss: 0.8988
Validation Loss: 1.4410 | Macro AUROC: nan
  - No Finding: 0.8895
  - Enlarged Cardiomediastinum: 0.5772
  - Cardiomegaly: 0.7660
  - Lung Opacity: 0.8983
  - Lung Lesion: 0.1545
  - Edema: 0.9087
  - Consolidation: 0.9189
  - Pneumonia: 0.6770
  - Atelectasis: 0.7855
  - Pneumothorax: 0.8084
  - Pleural Effusion: 0.9270
  - Pleural Other: 0.9914
  - Fracture: nan
  - Support Devices: 0.8193
Saving checkpoint to /kaggle/working/vit-chest-xray-full-labels-epoch-2...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Epoch 5/5 ---
Training Loss: 0.7934
Validation Loss: 1.4658 | Macro AUROC: nan
  - No Finding: 0.8904
  - Enlarged Cardiomediastinum: 0.4679
  - Cardiomegaly: 0.7692
  - Lung Opacity: 0.8844
  - Lung Lesion: 0.2275
  - Edema: 0.9041
  - Consolidation: 0.9139
  - Pneumonia: 0.6792
  - Atelectasis: 0.7923
  - Pneumothorax: 0.8512
  - Pleural Effusion: 0.9272
  - Pleural Other: 1.0000
  - Fracture: nan
  - Support Devices: 0.8483
Saving checkpoint to /kaggle/working/vit-chest-xray-full-labels-epoch-5...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]